# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oumaklaus/ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Lane:** Refresh / Content Opportunity Scoring (chosen in ML-02).
**ML task type: ranking / scoring** (with a supervised classifier underneath it).

Here's the reasoning. The decision is *"which pages should an editor review first?"* — that is an ordering question, not a yes/no verdict. So the deliverable is a **ranked queue**: every page gets a priority *score*, and the editor works down the list until their weekly capacity runs out. That is scoring, and evaluating an ordered shortlist is a ranking problem (the metric that matches it is **precision@K**, section 3).

Underneath, I'll train a **binary classifier** to estimate each page's probability of being a refresh candidate, and use that probability as the score. So the task is honestly *"a classifier used for ranking"* — the same shape the starter pipeline uses. It is **not** clustering (I have a target I care about, not just "what groups exist") and it is **not** plain classification for its own sake (a hard yes/no throws away the ordering the editor actually needs).

In [1]:
# Load the lane's slice of the starter data once, for the whole notebook.
import numpy as np
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print(f"rows (pages): {len(df):,}   columns: {df.shape[1]}   clients: {df['client_id'].nunique()}")

# The lane keeps pages that are old enough to judge and actually visible in search
# (mirrors the starter pipeline's filters: content_age_days >= 90 and impressions_90d > 0).
lane = df[(df["content_age_days"] >= 90) & (df["impressions_90d"] > 0)].copy()
print(f"lane slice after filters (age>=90 & impressions>0): {len(lane):,} pages")

rows (pages): 30,000   columns: 44   clients: 32
lane slice after filters (age>=90 & impressions>0): 30,000 pages


## 2. Target or proxy

**What I would predict:** for each page, the probability that it is a *refresh candidate* — a page whose search performance is slipping while it still has demand.

**The label I can build from the starter data is a PROXY, not a clean observed outcome.** The starter pipeline defines:

```text
is_declining_label = (trend_direction == "down")
```

`trend_direction` is a **defined rule**, not a future measurement: it's a bucket computed from *current-window* inputs (last-30-day vs previous-30-day impressions). Per `framing-ml-problems`, "the target must be observed, not defined" — a label that comes from a rule risks teaching the model the rule instead of the world. So I treat `is_declining_label` honestly as a **beginner proxy**: good enough to show the workflow end-to-end on the starter slice, but not the capstone target.

**Two consequences I commit to now:**
1. Because the label is *derived from* `trend_direction` (which is derived from `trend_pct`), those two columns can **never** be features — that's the leakage the starter notebook 02 demonstrates.
2. The stronger, genuinely-*observed* target for the capstone is forward-looking: build features from a prior window and measure decline in a **later** window (e.g. prior-90-day features -> impressions decline over the next 30 days), using the warehouse daily facts. I'll define that in the data contract (ML-04).

In [2]:
# Build the proxy target and sketch what the target column looks like.
lane["is_declining_label"] = (lane["trend_direction"] == "down").astype(int)

base_rate = lane["is_declining_label"].mean()
print(f"proxy positive rate (is_declining_label==1): {base_rate:.3f}  "
      f"({lane['is_declining_label'].sum():,} of {len(lane):,})")
print("\nlabel is well-balanced, so ROC-AUC and precision@K are both meaningful:")
print(lane["is_declining_label"].value_counts().rename({0: "not declining", 1: "declining"}).to_string())

# Quarantine list: columns that DERIVE the label and must never be features.
LEAKAGE_COLS = ["trend_direction", "trend_pct"]
print(f"\ncolumns quarantined from features (derive the label): {LEAKAGE_COLS}")

proxy positive rate (is_declining_label==1): 0.542  (16,262 of 30,000)

label is well-balanced, so ROC-AUC and precision@K are both meaningful:
is_declining_label
declining        16262
not declining    13738

columns quarantined from features (derive the label): ['trend_direction', 'trend_pct']


## 3. Success metric

**Primary metric: precision@K** (I'll report precision@50, and also @20).

Why this one, defended: the editor never works the whole list — they review the *top* of the queue until capacity runs out. So "good" means *the pages I put at the top really are the ones worth fixing*. Precision@50 answers exactly that: of the 50 pages the system ranks first, what fraction are true refresh candidates. Overall accuracy would be the wrong call — with a ~54% base rate, a model could look "accurate" while ordering the top of the list badly, which is the only part the editor sees.

**What number means good:** beat the **base rate** (~0.54 — precision if you picked 50 pages at random) *and* beat the **transparent rule baseline** I build in ML-07. The starter reference shows the bar is real: its rule baseline scored precision@50 = 0.240 on its slice while a random forest reached 0.740. My own numbers must be re-earned on my slice with client-holdout validation — I don't inherit those.

**Secondary metric: ROC-AUC**, as a sanity check on overall ranking quality (the label is balanced, so AUC is informative here). Precision@K decides; AUC is context.

In [3]:
# Show precision@K is computable TODAY, on a trivial baseline, so the metric is real.
def precision_at_k(scores, y, k):
    order = np.argsort(-np.asarray(scores, dtype=float))
    return float(np.asarray(y)[order[:k]].mean())

y = lane["is_declining_label"]

# A throwaway single-signal "baseline": rank by low CTR (a plausible refresh hint).
naive_score = -lane["ctr"].fillna(lane["ctr"].max())
for k in (20, 50):
    print(f"precision@{k:<2d} ranking by low-CTR: {precision_at_k(naive_score, y, k):.3f}")
print(f"base rate (random pick):        {y.mean():.3f}")
print("\n-> the metric runs today; ML-07 will build a real, explainable baseline to beat.")

precision@20 ranking by low-CTR: 0.550
precision@50 ranking by low-CTR: 0.620
base rate (random pick):        0.542

-> the metric runs today; ML-07 will build a real, explainable baseline to beat.


## 4. The unit of analysis, as a real dataframe

**One row = one page** (one pseudonymous `content_id`), summarised over the trailing-90-day window. Not a client, not a day, not a query. The output ranks these rows; the editor acts on one page at a time.

The cell below shows the grain literally: a handful of rows with the columns a reviewer would actually reason about, plus a check that `content_id` is unique (one row per page, no accidental duplication).

In [4]:
# Prove the grain: one row per page, and no duplicate content_ids.
dupes = lane["content_id"].duplicated().sum()
print(f"rows: {len(lane):,}   unique content_id: {lane['content_id'].nunique():,}   duplicate ids: {dupes}")
assert dupes == 0, "grain broken: content_id should be unique (one row per page)"

# What one row of the unit of analysis looks like (reviewer-facing columns only; no raw/private fields).
show_cols = ["content_id", "impressions_90d", "clicks_90d", "avg_position",
             "ctr", "days_since_last_update", "content_age_days",
             "word_count", "is_declining_label"]
lane[show_cols].head(5)

rows: 30,000   unique content_id: 30,000   duplicate ids: 0


,content_id,impressions_90d,clicks_90d,avg_position,ctr,days_since_last_update,content_age_days,word_count,is_declining_label
0,content_304f48230142,3803,29,10.6,0.76,20,187,3221.0,1
1,content_a1fb4e703a9e,15320,7,20.3,0.05,25,445,2481.0,1
2,content_9aa793d4d895,12581,11,36.5,0.09,20,141,3515.0,1
3,content_331d6c4de07b,11751,58,6.2,0.49,22,463,NaN,0
4,content_d99b7a2d90ca,19140,24,44.0,0.13,14,263,2803.0,1


## 5. Why ML beats a fixed rule here

A fixed rule (an if-statement like *"flag pages not updated in 180+ days"*) fails at this task for two concrete, measured reasons — both shown in the cell below:

1. **No single signal orders the queue.** The strongest single feature correlates only ~0.16 with the proxy label, and ranking by any one signal barely beats — sometimes *loses to* — the 54% base rate. The refresh signal lives in the *combination* of demand, position, freshness, CTR and engagement, not in any one column. That "many signals, tangled" pattern is exactly where `framing-ml-problems` says ML earns its place.
2. **A rule flags a bucket; it can't rank inside it.** A threshold rule labels a whole group "stale" and stops — it gives every page in the bucket the same verdict, so it can't tell the editor *which of those to open first*. A scoring model produces a continuous priority, so the top-K is genuinely ordered.

**But ML is not magic here, and I'll keep saying so:** the model outputs a *decision-support ranking* — a prioritised shortlist for a human reviewer — not an automated verdict and not proof that a refresh will work. The transparent rule baseline (ML-07) still matters: it's the honest bar the model has to clear, and if the model can't beat it, that's a finding, not a failure.

**The content action this supports:** the editor opens the ranked queue, and for each top page decides to **refresh / rewrite / expand**, **protect**, or **leave it** — guided by the reason codes attached to each page (e.g. `declining_with_demand`, `low_ctr_visible_page`).

In [5]:
# Evidence for claim 1: single signals are weak and don't cleanly order the queue.
def precision_at_50(scores, y):
    order = np.argsort(-np.asarray(scores, dtype=float))
    return float(np.asarray(y)[order[:50]].mean())

signals = {
    "days_since_last_update": lane["days_since_last_update"],
    "impressions_90d":        lane["impressions_90d"],
    "low ctr first":          -lane["ctr"].fillna(lane["ctr"].max()),
    "worse position first":   -lane["avg_position"].replace(0, np.nan).fillna(lane["avg_position"].max()),
}
print(f"base rate (random top-50): {y.mean():.3f}")
print("precision@50 for each SINGLE-signal ranking:")
for name, s in signals.items():
    print(f"  {name:24s}: {precision_at_50(s, y):.3f}")

# strongest single-signal correlation with the label -> weak, so combine signals (ML).
num = ["days_since_last_update", "avg_position", "impressions_90d", "ctr",
       "engagement_rate", "word_count", "content_age_days", "search_volume"]
corr = lane[num + ["is_declining_label"]].corr(numeric_only=True)["is_declining_label"].drop("is_declining_label")
print(f"\nstrongest single-signal |correlation| with the proxy label: {corr.abs().max():.3f}")
print("-> no single rule dominates; the signal is in the COMBINATION. That is the case for ML.")

base rate (random top-50): 0.542
precision@50 for each SINGLE-signal ranking:
  days_since_last_update  : 0.520
  impressions_90d         : 0.420
  low ctr first           : 0.620
  worse position first    : 0.180

strongest single-signal |correlation| with the proxy label: 0.164
-> no single rule dominates; the signal is in the COMBINATION. That is the case for ML.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.